# Reliability-Aware Scheduling of Home EV Charging Using Solar Forecast Confidence

### Interactive Demonstration and Exploration Notebook
This notebook demonstrates the end-to-end research workflow using the modular codebase in `src/`.


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys
sys.path.append("..")

# Load configuration
with open("../configs/default.json", "r") as f:
    config = json.load(f)

print("Project:", config["project_name"])
print("PV Capacity:", config["data"]["pv_capacity_kw"], "kW")
print("EV Battery Capacity:", config["ev"]["battery_capacity_kwh"], "kWh")


## 1. Load Dataset and Inspect Behind-the-Meter Data

In [ ]:
from src.data import load_dataset, create_splits

dataset = load_dataset(config)
splits = create_splits(dataset, config)

pv_df = dataset["pv"]
load_df = dataset["load"]
tariff_df = dataset["tariff"]

print(f"Total observations: {len(pv_df)} hours")
print(f"Splits: {list(splits.keys())}")

plt.figure(figsize=(12, 4))
plt.plot(pv_df.iloc[:72]["pv_generation_kw"], label="Rooftop PV (kW)", color="orange")
plt.plot(load_df.iloc[:72]["load_kw"], label="Household Load (kW)", color="blue")
plt.ylabel("Power (kW)")
plt.title("Sample 3-Day Behind-the-Meter Profile")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 2. Point Forecasting and Conformal Prediction Intervals

In [ ]:
from src.forecasting import fit_forecaster
from src.uncertainty import ConformalPredictor

train_df = pv_df.loc[splits["train"]["start"]:splits["train"]["end"]]
cal_df = pv_df.loc[splits["calibration"]["start"]:splits["calibration"]["end"]]
test_df = pv_df.loc[splits["test"]["start"]:splits["test"]["end"]]

f_ml = fit_forecaster(train_df, {"type": "ml", **config["forecasting"]})

# Conformal calibration
cal_preds = f_ml.predict(cal_df.index, pv_df)
cal_actuals = cal_df["pv_generation_kw"].values

conformal = ConformalPredictor(nominal_coverage=0.90)
conformal.calibrate(cal_actuals, cal_preds)
print(f"Calibrated 90% Conformal Quantile q_alpha: {conformal.q_alpha_:.4f} kW")

# Predict on first test day
test_day_idx = test_df.index[:24]
test_day_actual = test_df.iloc[:24]["pv_generation_kw"].values
test_day_forecast = f_ml.predict(test_day_idx, pv_df)
test_low, test_high = conformal.predict_interval(test_day_forecast, pv_capacity=5.0)

plt.figure(figsize=(10, 4))
plt.plot(test_day_actual, "b-o", label="Actual PV")
plt.plot(test_day_forecast, "r--", label="Forecast PV")
plt.fill_between(range(24), test_low, test_high, alpha=0.25, color="orange", label="90% Conformal Interval")
plt.xlabel("Hour of Day")
plt.ylabel("PV Generation (kW)")
plt.title("Test Day Forecast vs. Conformal Prediction Interval")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 3. Compare Charging Schedulers on a Sample Day

In [ ]:
from src.scheduler import solve_schedule
from src.simulation import simulate_day
from src.experiments import extract_day_data

test_date = test_df.index.date[0]
d_data = extract_day_data(dataset, test_date, config)

sched_inputs = {
    "pv_forecast": test_day_forecast,
    "load": d_data["load"],
    "tariff": d_data["tariff"],
    "ev_params": d_data["ev_params"],
    "actual_pv": test_day_actual,
    "config": {"dt": 1.0, "export_rate": 0.04}
}

controllers = ["fixed", "cheapest_hours", "forecast_driven", "conservative_fallback", "oracle"]
results = {}

for ctrl in controllers:
    sched = solve_schedule(sched_inputs, method=ctrl, config={"dt": 1.0})
    sim = simulate_day(sched, {"pv_actual": test_day_actual, "load": d_data["load"], "tariff": d_data["tariff"]}, d_data["ev_params"], {"dt": 1.0, "export_rate": 0.04})
    results[ctrl] = sim
    print(f"Controller {ctrl:22s} | Daily Cost: ${sim['total_cost']:.4f} | Deadline Met: {sim['deadline_met']}")

plt.figure(figsize=(10, 5))
for ctrl in controllers:
    plt.plot(results[ctrl]["charging_power"], label=ctrl, linewidth=2)
plt.xlabel("Hour of Day")
plt.ylabel("EV Charging Power (kW)")
plt.title("Charging Power Schedules by Controller")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 4. Aggregate Benchmark Results from Execution

In [ ]:
summary_table = pd.read_csv("../results/tables/table5_main_controller_comparison.csv", index_col=0)
summary_table[["mean_cost", "std_cost", "mean_grid_import", "mean_peak_grid", "deadline_success_rate"]]
